In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/telco_cleaned.csv')

# Sirf churned customers ka data (jinke paas Churn Reason hai)
churned_df = df[df['Churn Label'] == 'Yes']

# Har unique Churn Reason ki count dekho
print(churned_df['Churn Reason'].value_counts())
print("\nTotal unique reasons:", churned_df['Churn Reason'].nunique())

Churn Reason
Competitor had better devices                313
Competitor made better offer                 311
Attitude of support person                   220
Don't know                                   130
Competitor offered more data                 117
Competitor offered higher download speeds    100
Attitude of service provider                  94
Price too high                                78
Product dissatisfaction                       77
Network reliability                           72
Long distance charges                         64
Service dissatisfaction                       63
Moved                                         46
Extra data charges                            39
Limited range of services                     37
Poor expertise of online support              31
Lack of affordable download/upload speed      30
Lack of self-service on Website               29
Poor expertise of phone support               12
Deceased                                       6
Name: c

In [1]:
import os
import re

def parse_policy_file(filepath):
    """Parse a policy text file into structured fields (metadata + full text)"""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    def extract_field(field_name, text):
        pattern = rf"{field_name}:\s*(.+?)(?:\n\n|\n[A-Z])"
        match = re.search(pattern, text, re.DOTALL)
        return match.group(1).strip() if match else ""
    
    policy_id = extract_field("Policy ID", content)
    category = extract_field("Category", content)
    priority = extract_field("Priority Level", content)
    
    return {
        "policy_id": policy_id,
        "category": category,
        "priority": priority,
        "filename": os.path.basename(filepath),
        "full_text": content
    }

# Sab policy files load karo
policy_dir = '../knowledge_base'
policy_files = sorted([f for f in os.listdir(policy_dir) if f.endswith('.txt')])

policies = []
for fname in policy_files:
    filepath = os.path.join(policy_dir, fname)
    policy = parse_policy_file(filepath)
    policies.append(policy)

print(f"Loaded {len(policies)} policies:\n")
for p in policies:
    print(f"{p['policy_id']:15} | {p['category']:35} | {p['priority']}")

Loaded 11 policies:

P00_FALLBACK    | Non-Actionable / No Retention Policy Applicable | NOT APPLICABLE - no retention action should be recommended for this group
P01A            | Competitor-Related Churn            | HIGH
P01B            | Competitor-Related Churn            | HIGH
P01C            | Competitor-Related Churn            | MEDIUM-HIGH
P01D            | Competitor-Related Churn            | MEDIUM-HIGH
P02             | Support/Service Attitude Issues     | HIGH
P03             | Price & Charges Concerns            | MEDIUM-HIGH
P04             | Service Quality & Reliability       | HIGH
P05             | Technical Support Expertise Issues  | MEDIUM
P06             | Speed/Bandwidth Concerns            | MEDIUM
P07             | Digital Self-Service Gaps           | LOW-MEDIUM


In [2]:
from sentence_transformers import SentenceTransformer

# Lightweight, fast, aur achha quality wala model
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded!")
print("Embedding dimension:", model_embed.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\karrtik\anaconda3\envs\churn-assistant\lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\karrtik\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
Embedding dimension: 384


C:\Users\karrtik\AppData\Local\Temp\ipykernel_10004\8154889.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model_embed.get_sentence_embedding_dimension())


In [3]:
import numpy as np

# Har policy ke full_text ka embedding banao
policy_texts = [p['full_text'] for p in policies]
policy_embeddings = model_embed.encode(policy_texts, show_progress_bar=True)

print("Embeddings shape:", policy_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (11, 384)


In [4]:
import faiss

# Embedding dimension
dimension = policy_embeddings.shape[1]

# FAISS index banao - IndexFlatIP use karenge (cosine similarity ke liye, normalize karne ke baad)
# Pehle embeddings normalize karo taaki cosine similarity accurate ho
faiss.normalize_L2(policy_embeddings)

index = faiss.IndexFlatIP(dimension)  # IP = Inner Product (normalized vectors ke saath cosine similarity ke barabar)
index.add(policy_embeddings)

print("FAISS index banaya!")
print("Total vectors in index:", index.ntotal)

FAISS index banaya!
Total vectors in index: 11


In [5]:
def retrieve_policy(query, top_k=3, min_score=0.3):
    """
    Given a query, retrieve the top_k most relevant policies from the knowledge base.
    Returns results with similarity scores, filtered by min_score threshold.
    """
    # Query ko bhi embed aur normalize karo (same tarike se jaise policies ko kiya tha)
    query_embedding = model_embed.encode([query])
    faiss.normalize_L2(query_embedding)
    
    # Search karo
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if score >= min_score:
            results.append({
                'policy_id': policies[idx]['policy_id'],
                'category': policies[idx]['category'],
                'score': float(score),
                'filename': policies[idx]['filename']
            })
    
    return results

# Test karo ek simple query se
test_results = retrieve_policy("customer is on month-to-month contract with high monthly charges")
for r in test_results:
    print(f"{r['policy_id']} | Score: {r['score']:.3f} | {r['category']}")

P03 | Score: 0.527 | Price & Charges Concerns
P01B | Score: 0.480 | Competitor-Related Churn
P01A | Score: 0.437 | Competitor-Related Churn


In [6]:
# Test 1: Competitor ka explicit mention - P01D expect karte hain top match
print("=== Test 1: WITH competitor mention ===")
results_1 = retrieve_policy("customer on DSL says a competitor offers faster download speeds")
for r in results_1:
    print(f"{r['policy_id']} | Score: {r['score']:.3f} | {r['category']}")

print("\n=== Test 2: WITHOUT competitor mention ===")
# Test 2: Sirf apna plan expensive lagna, koi competitor mention nahi - P06 expect karte hain top match
results_2 = retrieve_policy("customer feels the higher speed internet plan is too expensive for them")
for r in results_2:
    print(f"{r['policy_id']} | Score: {r['score']:.3f} | {r['category']}")

=== Test 1: WITH competitor mention ===
P01D | Score: 0.715 | Competitor-Related Churn
P06 | Score: 0.643 | Speed/Bandwidth Concerns
P01C | Score: 0.499 | Competitor-Related Churn

=== Test 2: WITHOUT competitor mention ===
P06 | Score: 0.672 | Speed/Bandwidth Concerns
P01D | Score: 0.615 | Competitor-Related Churn
P01C | Score: 0.538 | Competitor-Related Churn


In [7]:
# Test 3: Ek irrelevant/random query - dekhte hain scores kitne low aate hain
print("=== Test 3: Irrelevant query (no real match expected) ===")
results_3 = retrieve_policy("what is the weather like today", min_score=0.0)  # min_score 0 rakha taaki sab dikhe
for r in results_3:
    print(f"{r['policy_id']} | Score: {r['score']:.3f} | {r['category']}")

=== Test 3: Irrelevant query (no real match expected) ===
P05 | Score: 0.052 | Technical Support Expertise Issues
P04 | Score: 0.032 | Service Quality & Reliability
P01D | Score: 0.027 | Competitor-Related Churn


In [8]:
def retrieve_policy(query, top_k=3, min_score=0.30):
    """
    Given a query, retrieve the top_k most relevant policies from the knowledge base.
    Returns empty list if no result meets the min_score threshold (triggers fallback behavior).
    """
    query_embedding = model_embed.encode([query])
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if score >= min_score:
            results.append({
                'policy_id': policies[idx]['policy_id'],
                'category': policies[idx]['category'],
                'score': float(score),
                'full_text': policies[idx]['full_text']
            })
    
    return results

print("Retrieval function finalized with min_score = 0.30")

Retrieval function finalized with min_score = 0.30


In [9]:
import joblib
import pandas as pd

# Module 3 se saved SHAP values load karo
shap_values = joblib.load('../models/shap_values.pkl')
X_test = pd.read_csv('../data/processed/X_test.csv')

def build_query_from_shap(customer_index, top_n=5):
    """
    Converts a customer's top SHAP-driving features into a natural language 
    query string suitable for RAG retrieval.
    """
    shap_vals = shap_values[customer_index]
    feature_vals = X_test.iloc[customer_index]
    
    # Top positive contributors nikalo (jo churn risk badha rahe hain)
    contributions = pd.DataFrame({
        'feature': X_test.columns,
        'value': feature_vals.values,
        'shap_value': shap_vals
    })
    top_factors = contributions[contributions['shap_value'] > 0].sort_values(
        'shap_value', ascending=False
    ).head(top_n)
    
    # Natural language mein convert karo
    query_parts = []
    for _, row in top_factors.iterrows():
        feature_name = row['feature']
        value = row['value']
        
        # One-hot encoded columns ko readable banao (e.g., "Contract_Two Year" -> "Contract is Two Year")
        if '_' in feature_name and value == True:
            base_feature, category = feature_name.rsplit('_', 1)
            query_parts.append(f"{base_feature} is {category}")
        elif '_' in feature_name and value == False:
            continue  # False wale one-hot columns skip karo (not useful in query)
        else:
            query_parts.append(f"{feature_name} is {value}")
    
    query = "Customer profile: " + ", ".join(query_parts)
    return query

# Test karo ek churned customer pe
churned_indices = np.where(y_test.values == 1)[0] if 'y_test' in dir() else None

In [10]:
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

churned_indices = np.where(y_test.values == 1)[0]
test_customer = churned_indices[0]

query = build_query_from_shap(test_customer)
print("Generated Query:")
print(query)

Generated Query:
Customer profile: Age is 65, Tenure in Months is 2, Total Long Distance Charges is 41.98


In [11]:
def build_query_from_shap(customer_index, top_n=6):
    """
    Converts a customer's top SHAP-driving features into a natural language 
    query string suitable for RAG retrieval.
    """
    shap_vals = shap_values[customer_index]
    feature_vals = X_test.iloc[customer_index]
    
    contributions = pd.DataFrame({
        'feature': X_test.columns,
        'value': feature_vals.values,
        'shap_value': shap_vals
    })
    top_factors = contributions[contributions['shap_value'] > 0].sort_values(
        'shap_value', ascending=False
    ).head(top_n)
    
    query_parts = []
    contract_flags = {}  # Contract_One Year aur Contract_Two Year track karne ke liye
    
    for _, row in top_factors.iterrows():
        feature_name = row['feature']
        value = row['value']
        
        if feature_name.startswith('Contract_'):
            category = feature_name.split('_', 1)[1]
            contract_flags[category] = value
            continue
        
        if '_' in feature_name:
            base_feature, category = feature_name.rsplit('_', 1)
            if value == True:
                query_parts.append(f"{base_feature} is {category}")
            else:
                query_parts.append(f"{base_feature} is not {category}")
        else:
            query_parts.append(f"{feature_name} is {value}")
    
    # Contract logic: agar One Year aur Two Year dono False/missing hain -> Month-to-Month
    if contract_flags:
        if not contract_flags.get('One Year', False) and not contract_flags.get('Two Year', False):
            query_parts.insert(0, "Contract is Month-to-Month")
        elif contract_flags.get('One Year', False):
            query_parts.insert(0, "Contract is One Year")
        elif contract_flags.get('Two Year', False):
            query_parts.insert(0, "Contract is Two Year")
    
    query = "Customer profile: " + ", ".join(query_parts)
    return query

# Dobara test karo
query = build_query_from_shap(test_customer)
print("Generated Query:")
print(query)

Generated Query:
Customer profile: Contract is Month-to-Month, Age is 65, Tenure in Months is 2, Total Long Distance Charges is 41.98, Payment Method is not Credit Card


In [12]:
# Is customer ke liye policy retrieve karo
results = retrieve_policy(query)

print("Query:", query)
print("\nRetrieved Policies:")
for r in results:
    print(f"{r['policy_id']} | Score: {r['score']:.3f} | {r['category']}")

Query: Customer profile: Contract is Month-to-Month, Age is 65, Tenure in Months is 2, Total Long Distance Charges is 41.98, Payment Method is not Credit Card

Retrieved Policies:
P03 | Score: 0.421 | Price & Charges Concerns
P01B | Score: 0.398 | Competitor-Related Churn
P01A | Score: 0.352 | Competitor-Related Churn


In [13]:
# Ek artificial low-signal query try karo - dekhte hain low scores aate hain ya nahi
test_query_weak = "Customer profile: Age is 45, Population is 25000"
weak_results = retrieve_policy(test_query_weak)

print("Query:", test_query_weak)
print("\nRetrieved Policies (with min_score=0.30 filter):")
if len(weak_results) == 0:
    print("No policies matched above threshold — FALLBACK should be used here.")
else:
    for r in weak_results:
        print(f"{r['policy_id']} | Score: {r['score']:.3f} | {r['category']}")

Query: Customer profile: Age is 45, Population is 25000

Retrieved Policies (with min_score=0.30 filter):
P04 | Score: 0.378 | Service Quality & Reliability
P02 | Score: 0.377 | Support/Service Attitude Issues
P07 | Score: 0.354 | Digital Self-Service Gaps


In [14]:
def retrieve_policy(query, top_k=3, min_score=0.40):
    """
    Updated threshold based on testing: 0.30 was too permissive, 
    allowing weak/generic queries to falsely match. 0.40 provides better separation.
    """
    query_embedding = model_embed.encode([query])
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if score >= min_score:
            results.append({
                'policy_id': policies[idx]['policy_id'],
                'category': policies[idx]['category'],
                'score': float(score),
                'full_text': policies[idx]['full_text']
            })
    
    return results

# Dobara test karo weak query pe naye threshold ke saath
weak_results = retrieve_policy("Customer profile: Age is 45, Population is 25000")
print("Weak query results with threshold 0.40:")
if len(weak_results) == 0:
    print("No policies matched — FALLBACK correctly triggered.")
else:
    for r in weak_results:
        print(f"{r['policy_id']} | Score: {r['score']:.3f}")

# Aur strong query pe bhi dobara confirm karo ki still match hoti hai
print("\nStrong query results with threshold 0.40:")
strong_results = retrieve_policy(query)  # humara pehla wala real churned customer query
for r in strong_results:
    print(f"{r['policy_id']} | Score: {r['score']:.3f}")

Weak query results with threshold 0.40:
No policies matched — FALLBACK correctly triggered.

Strong query results with threshold 0.40:
P03 | Score: 0.421


In [15]:
import pickle

# FAISS index save karo
faiss.write_index(index, '../models/policy_index.faiss')

# Policies metadata save karo (FAISS sirf vectors store karta hai, actual text/metadata alag se chahiye)
with open('../models/policies_metadata.pkl', 'wb') as f:
    pickle.dump(policies, f)

# Embedding model ka naam save karo (taaki Module 5/6 mein same model load ho)
with open('../models/embedding_model_name.txt', 'w') as f:
    f.write('all-MiniLM-L6-v2')

# Final threshold save karo
joblib.dump(0.40, '../models/rag_min_score.pkl')

print("FAISS index, policies metadata, embedding model name, aur threshold — sab saved!")

FAISS index, policies metadata, embedding model name, aur threshold — sab saved!


In [16]:
def build_llm_prompt(customer_index, retrieved_policies):
    """
    Builds a grounded prompt for the LLM, ensuring it only uses retrieved 
    policy information and explicitly handles the no-match (fallback) case.
    """
    shap_reasons = get_top_shap_reasons(customer_index, top_n=5)
    
    shap_context = "\n".join([
        f"- {row['feature']}: {row['value']} (impact score: {row['shap_value']:.3f})"
        for _, row in shap_reasons.iterrows()
    ])
    
    if len(retrieved_policies) == 0:
        policy_context = "NO RELEVANT POLICY FOUND (score below confidence threshold)."
        instruction = (
            "No confident policy match was found for this customer. "
            "Do NOT invent a retention recommendation. Instead, state clearly that "
            "no specific evidence-based policy applies, and suggest a general, "
            "low-risk action such as a personal check-in call to understand the "
            "customer's situation better."
        )
    else:
        policy_context = "\n\n".join([
            f"[{p['policy_id']}] {p['category']} (relevance score: {p['score']:.3f})\n{p['full_text']}"
            for p in retrieved_policies
        ])
        instruction = (
            "Using ONLY the retrieved policy information above, write a short, "
            "personalized retention recommendation for this customer. "
            "Do not invent actions, discounts, or numbers that are not present "
            "in the retrieved policy text. If the retrieved policies don't clearly "
            "apply to this customer's situation, say so honestly rather than "
            "forcing a recommendation."
        )
    
    prompt = f"""You are a customer retention assistant. A machine learning model has flagged this customer as high churn risk. Below is why (from SHAP feature analysis), and any matching retention policy retrieved from the company's knowledge base.

CUSTOMER'S TOP CHURN-RISK FACTORS (from SHAP):
{shap_context}

RETRIEVED RETENTION POLICY:
{policy_context}

INSTRUCTIONS: {instruction}
"""
    return prompt

# Test the prompt structure without calling the LLM yet
sample_prompt = build_llm_prompt(test_customer, strong_results)
print(sample_prompt)

NameError: name 'get_top_shap_reasons' is not defined

In [17]:
def get_top_shap_reasons(customer_index, top_n=5):
    """
    Given a customer's index in X_test, return their top N churn-driving factors
    (feature name, their value, and SHAP contribution).
    """
    shap_vals = shap_values[customer_index]
    feature_vals = X_test.iloc[customer_index]
    
    contributions = pd.DataFrame({
        'feature': X_test.columns,
        'value': feature_vals.values,
        'shap_value': shap_vals
    })
    
    top_positive = contributions.sort_values('shap_value', ascending=False).head(top_n)
    
    return top_positive

print("Function defined.")

Function defined.


In [18]:
sample_prompt = build_llm_prompt(test_customer, strong_results)
print(sample_prompt)

You are a customer retention assistant. A machine learning model has flagged this customer as high churn risk. Below is why (from SHAP feature analysis), and any matching retention policy retrieved from the company's knowledge base.

CUSTOMER'S TOP CHURN-RISK FACTORS (from SHAP):
- Age: 65 (impact score: 0.790)
- Contract_One Year: False (impact score: 0.726)
- Contract_Two Year: False (impact score: 0.671)
- Tenure in Months: 2 (impact score: 0.606)
- Total Long Distance Charges: 41.98 (impact score: 0.398)

RETRIEVED RETENTION POLICY:
[P03] Price & Charges Concerns (relevance score: 0.421)
Policy ID: P03
Category: Price & Charges Concerns
Churn Reason (exact dataset values): Price too high, Extra data charges, Long distance charges
Frequency: 181 cases (9.7% of all churned customers)
Priority Level: MEDIUM-HIGH
Related Policy: P01B (Competitor made better offer) - overlaps when the customer's price complaint is framed as a comparison to a named competitor rather than a standalone aff